# Notebook 24 — Relative Importance of Subject/Signal Quality vs Classifier Choice

Research question:

> What is the most important factor? Can we make a reasonable argument that subject quality is more important than classifier type?

This notebook quantifies that question using the higher-subject-count results already produced.

The analysis compares the relative contribution of:

1. classifier identity
2. subject identity
3. interpretable subject/signal quality
4. dataset/task condition

The notebook uses both evaluation settings:

- grouped within-subject decoding
- LOSO / cross-subject decoding

The main model set remains:

- LDA
- Logistic Regression
- EEGNet

No new classifiers are added here.

## Interpretation Goal

The goal is not to claim that classifier choice is irrelevant.

The goal is to test whether, in this project's datasets, subject-level signal quality explains more variation in decoding performance than the specific classifier used.

Useful evidence would include:

- between-subject AUC variability is larger than between-classifier variability
- the same subjects tend to be easy or difficult across classifiers
- subject-quality features explain additional variance beyond classifier identity
- classifier identity adds relatively little after accounting for dataset and subject/signal quality

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from scipy import stats

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

from oddball_eeg.config import get_experiment
from oddball_eeg.visualization import round_numeric_columns


ARTIFACT_ROOT = PROJECT_ROOT / "results" / "thesis_artifacts"

OUT_DIR = ARTIFACT_ROOT / "relative_importance_subject_quality_vs_classifier"
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUT_DIR)

In [ ]:
MAIN_EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

LOSO_EXPERIMENT_NAMES = [
    name.replace("_within_subject_", "_loso_")
    for name in MAIN_EXPERIMENT_NAMES
]

MODEL_ORDER = ["lda", "logreg", "eegnet"]

MODEL_LABELS = {
    "lda": "LDA",
    "logreg": "LogReg",
    "eegnet": "EEGNet",
}

print("Within-subject experiments:")
for name in MAIN_EXPERIMENT_NAMES:
    print("-", name)

print("\nLOSO experiments:")
for name in LOSO_EXPERIMENT_NAMES:
    print("-", name)

In [ ]:
def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


def safe_spearman(x, y):
    x = pd.Series(x)
    y = pd.Series(y)

    mask = x.notna() & y.notna()
    x = x[mask]
    y = y[mask]

    if len(x) < 5:
        return np.nan, np.nan, int(len(x))

    if x.nunique(dropna=True) < 3 or y.nunique(dropna=True) < 3:
        return np.nan, np.nan, int(len(x))

    rho, p = stats.spearmanr(x, y)
    return float(rho), float(p), int(len(x))


def normalize_model_name(model):
    m = str(model).strip().lower()

    aliases = {
        "lda": "lda",
        "linear_discriminant_analysis": "lda",
        "logreg": "logreg",
        "logistic_regression": "logreg",
        "logistic regression": "logreg",
        "lr": "logreg",
        "eegnet": "eegnet",
        "current_eegnet": "eegnet",
    }

    return aliases.get(m, m)


def make_dataset_label(row):
    dataset = str(row.get("dataset", "")).strip()
    event_preset = str(row.get("event_preset", "")).strip()

    if "dataset_label" in row and pd.notna(row.get("dataset_label")):
        existing = str(row.get("dataset_label")).strip()
        if existing and existing.lower() not in ["nan", "none"]:
            return existing

    if dataset == "ds006480":
        if event_preset and event_preset.lower() not in ["nan", "none", ""]:
            return f"{dataset}_{event_preset}"
        return "ds006480_pooled"

    return dataset


def normalize_subject_id(subject, dataset_label=None, dataset=None):
    if pd.isna(subject):
        return np.nan

    s = str(subject).strip()

    if s.startswith("sub-"):
        s = s.replace("sub-", "", 1)

    if s.endswith(".0"):
        s = s[:-2]

    label = str(dataset_label or dataset or "")

    if s.isdigit():
        if label.startswith("ds003061") or label.startswith("ds005863"):
            return s.zfill(3)
        if label.startswith("ds006480"):
            return s.zfill(2)

    return s


def zscore_within_group(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    mu = s.mean()
    sd = s.std(ddof=0)

    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=series.index)

    return (s - mu) / sd


def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

## Load Within-Subject Subject-Level AUCs

Each row should represent:

- dataset
- subject
- model
- within-subject AUC

These are loaded from Notebook 02 outputs.

In [ ]:
def find_within_subject_summary_file(cfg):
    base = Path(cfg.result_dir())

    preferred_names = [
        "subject_summary_across_seeds.csv",
        "subject_seed_summary.csv",
    ]

    for filename in preferred_names:
        matches = sorted(base.rglob(filename))
        if matches:
            return matches[0]

    return None


def load_one_within_subject_auc(experiment_name):
    cfg = get_experiment(experiment_name)
    path = find_within_subject_summary_file(cfg)

    if path is None:
        raise FileNotFoundError(
            f"Could not find subject summary for {experiment_name} under {cfg.result_dir()}"
        )

    df = pd.read_csv(path)

    df["experiment_name"] = experiment_name
    df["evaluation_setting"] = "within_subject"
    df["dataset"] = cfg.dataset.dataset
    df["task"] = cfg.dataset.task
    df["event_preset"] = cfg.dataset.event_preset
    df["run_tag"] = cfg.run_tag

    if "dataset_label" not in df.columns:
        df["dataset_label"] = df.apply(make_dataset_label, axis=1)
    else:
        df["dataset_label"] = df.apply(make_dataset_label, axis=1)

    if "subject" not in df.columns:
        raise ValueError(f"No subject column found in {path}")

    df["subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(df["subject"], df["dataset_label"], df["dataset"])
    ]

    if "model" not in df.columns:
        raise ValueError(f"No model column found in {path}")

    df["model"] = df["model"].map(normalize_model_name)

    if "auc" not in df.columns:
        auc_candidates = [
            "auc_seed_mean",
            "auc_mean",
            "mean_auc",
            "subject_mean_auc",
            "roc_auc",
        ]

        found_auc = None
        for col in auc_candidates:
            if col in df.columns:
                found_auc = col
                break

        if found_auc is None:
            raise ValueError(
                f"No AUC column found in {path}. Columns: {list(df.columns)}"
            )

        df["auc"] = df[found_auc]

    if "auc_std" not in df.columns:
        std_candidates = [
            "auc_seed_std",
            "auc_std",
            "std_auc",
        ]

        found_std = None
        for col in std_candidates:
            if col in df.columns:
                found_std = col
                break

        df["auc_std"] = df[found_std] if found_std is not None else np.nan

    df = df[df["model"].isin(MODEL_ORDER)].copy()

    keep_cols = [
        "evaluation_setting",
        "dataset_label",
        "dataset",
        "task",
        "event_preset",
        "subject",
        "model",
        "auc",
        "auc_std",
        "run_tag",
        "experiment_name",
    ]

    return df[keep_cols].copy()


within_auc_dfs = []

for name in MAIN_EXPERIMENT_NAMES:
    loaded = load_one_within_subject_auc(name)
    within_auc_dfs.append(loaded)

within_auc_df = pd.concat(within_auc_dfs, ignore_index=True)

within_auc_df["auc"] = pd.to_numeric(within_auc_df["auc"], errors="coerce")

within_auc_df = (
    within_auc_df
    .dropna(subset=["dataset_label", "subject", "model", "auc"])
    .drop_duplicates(subset=["evaluation_setting", "dataset_label", "subject", "model"], keep="last")
    .sort_values(["dataset_label", "subject", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    within_auc_df,
    TABLE_DIR / "within_subject_model_subject_auc_table.csv",
)

print("Within-subject AUC table:", within_auc_df.shape)
display(round_numeric_columns(within_auc_df.head(20), 4))

## Load LOSO Subject-Level AUCs

Each row should represent:

- dataset
- held-out subject
- model
- LOSO AUC

These are loaded from Notebook 23 if available, or directly from Notebook 04 outputs.

In [ ]:
def find_loso_subject_file(cfg):
    base = Path(cfg.result_dir())

    preferred_names = [
        "loso_subject_auc.csv",
        "loso_subject_summary_across_seeds.csv",
    ]

    for filename in preferred_names:
        matches = sorted(base.rglob(filename))
        if matches:
            return matches[0]

    return None


def load_loso_from_notebook23_if_available():
    path = (
            ARTIFACT_ROOT
            / "loso_susceptibility_fdr"
            / "tables"
            / "loso_subject_auc.csv"
    )

    if not path.exists():
        return None

    df = pd.read_csv(path)

    df["evaluation_setting"] = "loso"

    if "heldout_subject" not in df.columns:
        if "subject" in df.columns:
            df["heldout_subject"] = df["subject"]
        else:
            raise ValueError(f"No subject/heldout_subject column found in {path}")

    if "subject" not in df.columns:
        df["subject"] = df["heldout_subject"]

    df["dataset_label"] = df.apply(make_dataset_label, axis=1)

    df["subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(df["subject"], df["dataset_label"], df["dataset"])
    ]

    df["heldout_subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(df["heldout_subject"], df["dataset_label"], df["dataset"])
    ]

    df["model"] = df["model"].map(normalize_model_name)

    if "auc" not in df.columns:
        if "loso_auc" in df.columns:
            df["auc"] = df["loso_auc"]
        else:
            raise ValueError(f"No LOSO AUC column found in {path}")

    if "auc_std" not in df.columns:
        if "loso_auc_std" in df.columns:
            df["auc_std"] = df["loso_auc_std"]
        else:
            df["auc_std"] = np.nan

    df = df[df["model"].isin(MODEL_ORDER)].copy()

    keep_cols = [
        "evaluation_setting",
        "dataset_label",
        "dataset",
        "task",
        "event_preset",
        "subject",
        "heldout_subject",
        "model",
        "auc",
        "auc_std",
        "run_tag",
        "loso_experiment",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    return df[keep_cols].copy()


def load_one_loso_subject_auc(experiment_name):
    cfg = get_experiment(experiment_name)
    path = find_loso_subject_file(cfg)

    if path is None:
        raise FileNotFoundError(
            f"Could not find LOSO subject summary for {experiment_name} under {cfg.result_dir()}"
        )

    df = pd.read_csv(path)

    df["evaluation_setting"] = "loso"
    df["dataset"] = cfg.dataset.dataset
    df["task"] = cfg.dataset.task
    df["event_preset"] = cfg.dataset.event_preset
    df["run_tag"] = cfg.run_tag
    df["loso_experiment"] = experiment_name

    if "dataset_label" not in df.columns:
        df["dataset_label"] = df.apply(make_dataset_label, axis=1)
    else:
        df["dataset_label"] = df.apply(make_dataset_label, axis=1)

    if "heldout_subject" not in df.columns:
        if "subject" in df.columns:
            df["heldout_subject"] = df["subject"]
        else:
            raise ValueError(f"No subject/heldout_subject column found in {path}")

    if "subject" not in df.columns:
        df["subject"] = df["heldout_subject"]

    df["subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(df["subject"], df["dataset_label"], df["dataset"])
    ]

    df["heldout_subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(df["heldout_subject"], df["dataset_label"], df["dataset"])
    ]

    df["model"] = df["model"].map(normalize_model_name)

    if "auc" not in df.columns:
        auc_candidates = [
            "loso_auc",
            "auc_seed_mean",
            "auc_mean",
            "mean_auc",
            "auc",
        ]

        found_auc = None
        for col in auc_candidates:
            if col in df.columns:
                found_auc = col
                break

        if found_auc is None:
            raise ValueError(
                f"No LOSO AUC column found in {path}. Columns: {list(df.columns)}"
            )

        df["auc"] = df[found_auc]

    if "auc_std" not in df.columns:
        std_candidates = [
            "loso_auc_std",
            "auc_seed_std",
            "auc_std",
            "std_auc",
        ]

        found_std = None
        for col in std_candidates:
            if col in df.columns:
                found_std = col
                break

        df["auc_std"] = df[found_std] if found_std is not None else np.nan

    df = df[df["model"].isin(MODEL_ORDER)].copy()

    keep_cols = [
        "evaluation_setting",
        "dataset_label",
        "dataset",
        "task",
        "event_preset",
        "subject",
        "heldout_subject",
        "model",
        "auc",
        "auc_std",
        "run_tag",
        "loso_experiment",
    ]

    return df[keep_cols].copy()


loso_auc_df = load_loso_from_notebook23_if_available()

if loso_auc_df is None:
    loso_auc_dfs = []
    for name in LOSO_EXPERIMENT_NAMES:
        loaded = load_one_loso_subject_auc(name)
        loso_auc_dfs.append(loaded)
    loso_auc_df = pd.concat(loso_auc_dfs, ignore_index=True)

loso_auc_df["auc"] = pd.to_numeric(loso_auc_df["auc"], errors="coerce")

loso_auc_df = (
    loso_auc_df
    .dropna(subset=["dataset_label", "subject", "model", "auc"])
    .drop_duplicates(subset=["evaluation_setting", "dataset_label", "subject", "model"], keep="last")
    .sort_values(["dataset_label", "subject", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    loso_auc_df,
    TABLE_DIR / "loso_model_subject_auc_table.csv",
)

print("LOSO AUC table:", loso_auc_df.shape)
display(round_numeric_columns(loso_auc_df.head(20), 4))

In [ ]:
auc_long_df = pd.concat(
    [
        within_auc_df,
        loso_auc_df,
    ],
    ignore_index=True,
)

auc_long_df["model"] = pd.Categorical(
    auc_long_df["model"],
    categories=MODEL_ORDER,
    ordered=True,
)

auc_long_df["subject_key"] = (
        auc_long_df["dataset_label"].astype(str)
        + "::"
        + auc_long_df["subject"].astype(str)
)

auc_long_df = (
    auc_long_df
    .sort_values(["evaluation_setting", "dataset_label", "subject", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    auc_long_df,
    TABLE_DIR / "relative_importance_subject_model_auc_table.csv",
)

print("Combined AUC table:", auc_long_df.shape)

display(
    round_numeric_columns(
        auc_long_df
        .groupby(["evaluation_setting", "dataset_label", "model"], observed=False)
        .agg(
            n_subjects=("subject", "nunique"),
            mean_auc=("auc", "mean"),
            std_auc=("auc", "std"),
            min_auc=("auc", "min"),
            max_auc=("auc", "max"),
        )
        .reset_index(),
        4,
    )
)

## Load Subject Raw/ERP/Power Features

The subject-quality analysis uses the raw/ERP/power feature table created earlier.

The goal is to construct a compact, interpretable **subject quality index**.

Higher subject quality should mean:

- stronger ERP/P300 separability
- lower baseline/post-stimulus noise

In [ ]:
raw_feature_candidates = [
    ARTIFACT_ROOT / "loso_susceptibility_fdr" / "tables" / "subject_raw_erp_power_features_normalized.csv",
    ARTIFACT_ROOT / "classification_susceptibility" / "tables" / "subject_raw_eeg_features.csv",
    ARTIFACT_ROOT / "classification_susceptibility" / "tables" / "subject_raw_erp_power_features.csv",
    ARTIFACT_ROOT / "multimodel_classification_susceptibility" / "tables" / "subject_raw_eeg_features.csv",
    ARTIFACT_ROOT / "multimodel_classification_susceptibility" / "tables" / "subject_raw_erp_power_features.csv",
]

raw_feature_path = find_first_existing(raw_feature_candidates)

if raw_feature_path is None:
    matches = sorted((PROJECT_ROOT / "results").rglob("subject_raw*features*.csv"))
    if matches:
        raw_feature_path = matches[0]

if raw_feature_path is None:
    raise FileNotFoundError(
        "Could not find subject raw/ERP/power feature table. "
        "Run Notebook 17/17b or Notebook 23 first."
    )

subject_features_df = pd.read_csv(raw_feature_path)

print("Subject feature table:", raw_feature_path)
print("Shape:", subject_features_df.shape)

display(subject_features_df.head())

In [ ]:
def normalize_subject_feature_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "dataset_label" not in out.columns:
        if "dataset" not in out.columns:
            raise ValueError("Feature table has no dataset or dataset_label column.")
        out["dataset_label"] = out.apply(make_dataset_label, axis=1)
    else:
        out["dataset_label"] = out.apply(make_dataset_label, axis=1)

    if "subject" not in out.columns:
        candidate_subject_cols = [
            "heldout_subject",
            "subject_id",
            "sub",
        ]

        found_subject = None
        for col in candidate_subject_cols:
            if col in out.columns:
                found_subject = col
                break

        if found_subject is None:
            raise ValueError("Feature table has no subject column.")

        out["subject"] = out[found_subject]

    dataset_col = out["dataset"] if "dataset" in out.columns else out["dataset_label"]

    out["subject"] = [
        normalize_subject_id(s, dataset_label=dl, dataset=ds)
        for s, dl, ds in zip(out["subject"], out["dataset_label"], dataset_col)
    ]

    return out


subject_features_df = normalize_subject_feature_table(subject_features_df)

subject_features_df = (
    subject_features_df
    .drop_duplicates(subset=["dataset_label", "subject"], keep="last")
    .sort_values(["dataset_label", "subject"])
    .reset_index(drop=True)
)

subject_features_df["subject_key"] = (
        subject_features_df["dataset_label"].astype(str)
        + "::"
        + subject_features_df["subject"].astype(str)
)

save_dataframe(
    subject_features_df,
    TABLE_DIR / "subject_raw_erp_power_features_for_relative_importance.csv",
)

print("Normalized subject feature table:", subject_features_df.shape)
display(subject_features_df[["dataset_label", "subject", "subject_key"]].head(20))

## Build Subject Quality Index

This index is deliberately simple and interpretable.

It combines:

1. ERP/P300 separability features with positive sign
2. noise features with negative sign

All features are z-scored within dataset before averaging.

This prevents one dataset's feature scale from dominating the index.

In [ ]:
IDENTIFIER_COLS = {
    "dataset",
    "dataset_label",
    "task",
    "event_preset",
    "subject",
    "subject_key",
    "heldout_subject",
    "model",
    "auc",
    "auc_std",
    "run_tag",
    "experiment_name",
    "loso_experiment",
}


def is_numeric_feature(df, col):
    if col in IDENTIFIER_COLS:
        return False

    values = pd.to_numeric(df[col], errors="coerce")
    return values.notna().sum() >= 10 and values.nunique(dropna=True) >= 3


numeric_feature_cols = [
    col for col in subject_features_df.columns
    if is_numeric_feature(subject_features_df, col)
]


def is_positive_quality_feature(col):
    c = col.lower()

    if "latency" in c:
        return False

    if "baseline" in c and any(k in c for k in ["rms", "std", "noise"]):
        return False

    if "post" in c and any(k in c for k in ["rms", "std", "noise"]):
        return False

    positive_keywords = [
        "p300",
        "erp_snr",
        "snr",
        "cohen",
        "target_standard",
        "target_minus_standard",
        "mean_diff",
        "peak",
        "rms_diff",
    ]

    return any(k in c for k in positive_keywords)


def is_negative_quality_feature(col):
    c = col.lower()

    if "p300" in c or "cohen" in c or "snr" in c:
        return False

    negative_patterns = [
        ("baseline", "rms"),
        ("baseline", "std"),
        ("baseline", "noise"),
        ("post", "rms"),
        ("post", "std"),
        ("post", "noise"),
        ("noise", ""),
    ]

    for a, b in negative_patterns:
        if a in c and (b == "" or b in c):
            return True

    return False


positive_quality_cols = [
    col for col in numeric_feature_cols
    if is_positive_quality_feature(col)
]

negative_quality_cols = [
    col for col in numeric_feature_cols
    if is_negative_quality_feature(col)
]

print("Positive quality features:")
for col in positive_quality_cols:
    print("-", col)

print("\nNegative quality/noise features:")
for col in negative_quality_cols:
    print("-", col)

quality_feature_selection_df = pd.DataFrame({
    "feature": positive_quality_cols + negative_quality_cols,
    "orientation": (
            ["positive_quality"] * len(positive_quality_cols)
            + ["negative_quality_noise"] * len(negative_quality_cols)
    ),
})

save_dataframe(
    quality_feature_selection_df,
    TABLE_DIR / "subject_quality_index_selected_features.csv",
)

display(quality_feature_selection_df)

In [ ]:
quality_df = subject_features_df[[
    "dataset_label",
    "subject",
    "subject_key",
]].copy()

quality_component_cols = []

for col in positive_quality_cols:
    z_col = f"z_pos__{col}"
    quality_df[z_col] = (
        subject_features_df
        .groupby("dataset_label")[col]
        .transform(zscore_within_group)
    )
    quality_component_cols.append(z_col)

for col in negative_quality_cols:
    z_col = f"z_neg_reversed__{col}"
    quality_df[z_col] = -(
        subject_features_df
        .groupby("dataset_label")[col]
        .transform(zscore_within_group)
    )
    quality_component_cols.append(z_col)

if not quality_component_cols:
    raise RuntimeError(
        "No quality features were selected. "
        "Inspect feature column names and adjust the keyword rules."
    )

quality_df["subject_quality_index"] = quality_df[quality_component_cols].mean(
    axis=1,
    skipna=True,
)

quality_df["n_quality_components_available"] = quality_df[quality_component_cols].notna().sum(axis=1)

quality_df = (
    quality_df
    .sort_values(["dataset_label", "subject"])
    .reset_index(drop=True)
)

save_dataframe(
    quality_df,
    TABLE_DIR / "subject_quality_index.csv",
)

print("Quality index table:", quality_df.shape)

display(
    round_numeric_columns(
        quality_df[[
            "dataset_label",
            "subject",
            "subject_quality_index",
            "n_quality_components_available",
        ]].head(30),
        4,
    )
)

In [ ]:
quality_summary_df = (
    quality_df
    .groupby("dataset_label", as_index=False)
    .agg(
        n_subjects=("subject", "nunique"),
        mean_quality=("subject_quality_index", "mean"),
        std_quality=("subject_quality_index", "std"),
        min_quality=("subject_quality_index", "min"),
        max_quality=("subject_quality_index", "max"),
        mean_n_components=("n_quality_components_available", "mean"),
    )
)

save_dataframe(
    quality_summary_df,
    TABLE_DIR / "subject_quality_index_summary_by_dataset.csv",
)

display(round_numeric_columns(quality_summary_df, 4))

In [ ]:
auc_quality_df = auc_long_df.merge(
    quality_df[[
        "dataset_label",
        "subject",
        "subject_key",
        "subject_quality_index",
        "n_quality_components_available",
    ]],
    on=["dataset_label", "subject", "subject_key"],
    how="left",
)

merge_status_df = (
    auc_quality_df
    .assign(has_quality=lambda d: d["subject_quality_index"].notna())
    .groupby(["evaluation_setting", "dataset_label", "model"], observed=False, as_index=False)
    .agg(
        n_rows=("auc", "count"),
        n_subjects=("subject", "nunique"),
        n_with_quality=("has_quality", "sum"),
    )
)

save_dataframe(
    auc_quality_df,
    TABLE_DIR / "auc_with_subject_quality_index.csv",
)

save_dataframe(
    merge_status_df,
    TABLE_DIR / "quality_index_merge_status.csv",
)

display(merge_status_df)

print("Merged AUC + quality table:", auc_quality_df.shape)
display(round_numeric_columns(auc_quality_df.head(20), 4))

## Analysis 1 — Subject Variability vs Classifier Variability

This section compares:

- variability across subjects
- variability across classifiers

If subject-to-subject variability is much larger than classifier-to-classifier variability, this supports the argument that subject/signal quality may be more important than classifier identity.

In [ ]:
spread_rows = []

for setting, setting_df in auc_quality_df.groupby("evaluation_setting"):
    for dataset_label, ds_df in setting_df.groupby("dataset_label"):
        # Mean AUC per subject across classifiers
        subject_mean = (
            ds_df
            .groupby("subject", as_index=False)
            .agg(subject_mean_auc=("auc", "mean"))
        )

        # Mean AUC per classifier across subjects
        model_mean = (
            ds_df
            .groupby("model", observed=False, as_index=False)
            .agg(model_mean_auc=("auc", "mean"))
        )

        # Within-subject classifier spread
        subject_model_range = (
            ds_df
            .groupby("subject", as_index=False)
            .agg(
                subject_classifier_range=("auc", lambda x: float(np.nanmax(x) - np.nanmin(x))),
                subject_classifier_sd=("auc", "std"),
            )
        )

        subject_sd = subject_mean["subject_mean_auc"].std(ddof=1)
        classifier_sd = model_mean["model_mean_auc"].std(ddof=1)

        subject_range = subject_mean["subject_mean_auc"].max() - subject_mean["subject_mean_auc"].min()
        classifier_range = model_mean["model_mean_auc"].max() - model_mean["model_mean_auc"].min()

        spread_rows.append({
            "evaluation_setting": setting,
            "dataset_label": dataset_label,
            "n_subjects": subject_mean["subject"].nunique(),
            "n_models": model_mean["model"].nunique(),
            "subject_mean_auc_sd": subject_sd,
            "classifier_mean_auc_sd": classifier_sd,
            "subject_mean_auc_range": subject_range,
            "classifier_mean_auc_range": classifier_range,
            "sd_ratio_subject_over_classifier": (
                subject_sd / classifier_sd if classifier_sd and np.isfinite(classifier_sd) and classifier_sd > 0 else np.nan
            ),
            "range_ratio_subject_over_classifier": (
                subject_range / classifier_range if classifier_range and np.isfinite(classifier_range) and classifier_range > 0 else np.nan
            ),
            "mean_within_subject_classifier_range": subject_model_range["subject_classifier_range"].mean(),
            "median_within_subject_classifier_range": subject_model_range["subject_classifier_range"].median(),
            "mean_within_subject_classifier_sd": subject_model_range["subject_classifier_sd"].mean(),
        })

spread_df = pd.DataFrame(spread_rows)

save_dataframe(
    spread_df,
    TABLE_DIR / "subject_vs_classifier_spread_summary.csv",
)

display(round_numeric_columns(spread_df, 4))

In [ ]:
overall_spread_rows = []

for setting, setting_df in auc_quality_df.groupby("evaluation_setting"):
    # Dataset-zscore AUC first so cross-dataset level differences do not dominate
    temp = setting_df.copy()
    temp["auc_z"] = temp.groupby("dataset_label")["auc"].transform(zscore_within_group)

    subject_mean = (
        temp
        .groupby("subject_key", as_index=False)
        .agg(subject_mean_auc_z=("auc_z", "mean"))
    )

    model_mean = (
        temp
        .groupby("model", observed=False, as_index=False)
        .agg(model_mean_auc_z=("auc_z", "mean"))
    )

    subject_model_range = (
        temp
        .groupby("subject_key", as_index=False)
        .agg(
            subject_classifier_range_z=("auc_z", lambda x: float(np.nanmax(x) - np.nanmin(x))),
            subject_classifier_sd_z=("auc_z", "std"),
        )
    )

    subject_sd = subject_mean["subject_mean_auc_z"].std(ddof=1)
    classifier_sd = model_mean["model_mean_auc_z"].std(ddof=1)

    subject_range = subject_mean["subject_mean_auc_z"].max() - subject_mean["subject_mean_auc_z"].min()
    classifier_range = model_mean["model_mean_auc_z"].max() - model_mean["model_mean_auc_z"].min()

    overall_spread_rows.append({
        "evaluation_setting": setting,
        "n_subjects": subject_mean["subject_key"].nunique(),
        "n_models": model_mean["model"].nunique(),
        "subject_auc_z_sd": subject_sd,
        "classifier_auc_z_sd": classifier_sd,
        "subject_auc_z_range": subject_range,
        "classifier_auc_z_range": classifier_range,
        "sd_ratio_subject_over_classifier": (
            subject_sd / classifier_sd if classifier_sd and np.isfinite(classifier_sd) and classifier_sd > 0 else np.nan
        ),
        "range_ratio_subject_over_classifier": (
            subject_range / classifier_range if classifier_range and np.isfinite(classifier_range) and classifier_range > 0 else np.nan
        ),
        "mean_within_subject_classifier_range_z": subject_model_range["subject_classifier_range_z"].mean(),
        "median_within_subject_classifier_range_z": subject_model_range["subject_classifier_range_z"].median(),
    })

overall_spread_df = pd.DataFrame(overall_spread_rows)

save_dataframe(
    overall_spread_df,
    TABLE_DIR / "overall_subject_vs_classifier_spread_zscored_summary.csv",
)

display(round_numeric_columns(overall_spread_df, 4))

## Analysis 2 — Are Easy and Difficult Subjects Consistent Across Classifiers?

If subject AUCs are correlated across classifiers, then the same subjects tend to be easy or difficult regardless of classifier.

This supports the argument that subject/signal quality is a major driver of decoding performance.

In [ ]:
pair_rows = []

model_pairs = [
    ("lda", "logreg"),
    ("lda", "eegnet"),
    ("logreg", "eegnet"),
]

for setting, setting_df in auc_quality_df.groupby("evaluation_setting"):
    for dataset_label, ds_df in setting_df.groupby("dataset_label"):
        pivot = ds_df.pivot_table(
            index="subject",
            columns="model",
            values="auc",
            aggfunc="mean",
            observed=False,
        )

        for m1, m2 in model_pairs:
            if m1 not in pivot.columns or m2 not in pivot.columns:
                rho, p, n = np.nan, np.nan, 0
            else:
                rho, p, n = safe_spearman(pivot[m1], pivot[m2])

            pair_rows.append({
                "evaluation_setting": setting,
                "dataset_label": dataset_label,
                "model_1": m1,
                "model_2": m2,
                "n_subjects": n,
                "spearman_rho": rho,
                "p_value": p,
            })

cross_classifier_corr_df = pd.DataFrame(pair_rows)

save_dataframe(
    cross_classifier_corr_df,
    TABLE_DIR / "cross_classifier_subject_auc_correlations.csv",
)

display(round_numeric_columns(cross_classifier_corr_df, 4))

In [ ]:
cross_classifier_corr_summary_df = (
    cross_classifier_corr_df
    .groupby("evaluation_setting", as_index=False)
    .agg(
        n_correlations=("spearman_rho", lambda x: x.notna().sum()),
        mean_spearman_rho=("spearman_rho", "mean"),
        median_spearman_rho=("spearman_rho", "median"),
        min_spearman_rho=("spearman_rho", "min"),
        max_spearman_rho=("spearman_rho", "max"),
    )
)

save_dataframe(
    cross_classifier_corr_summary_df,
    TABLE_DIR / "cross_classifier_subject_auc_correlation_summary.csv",
)

display(round_numeric_columns(cross_classifier_corr_summary_df, 4))

## Analysis 3 — Quality Index vs AUC

This section checks whether the interpretable subject quality index is associated with AUC.

This is run separately for:

- within-subject decoding
- LOSO decoding

and separately for each model.

In [ ]:
quality_corr_rows = []

for setting, setting_df in auc_quality_df.groupby("evaluation_setting"):
    for model, model_df in setting_df.groupby("model", observed=False):
        rho, p, n = safe_spearman(
            model_df["subject_quality_index"],
            model_df["auc"],
        )

        quality_corr_rows.append({
            "evaluation_setting": setting,
            "dataset_label": "grand_raw",
            "model": model,
            "n_subjects": n,
            "spearman_rho_quality_vs_auc": rho,
            "p_value": p,
        })

        # Dataset-zscored version
        temp = model_df.copy()
        temp["quality_z"] = temp.groupby("dataset_label")["subject_quality_index"].transform(zscore_within_group)
        temp["auc_z"] = temp.groupby("dataset_label")["auc"].transform(zscore_within_group)

        rho_z, p_z, n_z = safe_spearman(
            temp["quality_z"],
            temp["auc_z"],
        )

        quality_corr_rows.append({
            "evaluation_setting": setting,
            "dataset_label": "grand_dataset_zscored",
            "model": model,
            "n_subjects": n_z,
            "spearman_rho_quality_vs_auc": rho_z,
            "p_value": p_z,
        })

        # Dataset-specific
        for dataset_label, ds_df in model_df.groupby("dataset_label"):
            rho_ds, p_ds, n_ds = safe_spearman(
                ds_df["subject_quality_index"],
                ds_df["auc"],
            )

            quality_corr_rows.append({
                "evaluation_setting": setting,
                "dataset_label": dataset_label,
                "model": model,
                "n_subjects": n_ds,
                "spearman_rho_quality_vs_auc": rho_ds,
                "p_value": p_ds,
            })

quality_auc_corr_df = pd.DataFrame(quality_corr_rows)

save_dataframe(
    quality_auc_corr_df,
    TABLE_DIR / "subject_quality_index_auc_correlations.csv",
)

display(
    round_numeric_columns(
        quality_auc_corr_df.sort_values(
            ["evaluation_setting", "dataset_label", "model"]
        ),
        4,
    )
)

## Analysis 4 — Incremental R²

This is the most direct answer to the research question.

We compare simple regression models predicting AUC:

- dataset only
- classifier only
- subject quality only
- dataset + classifier
- dataset + subject quality
- dataset + classifier + subject quality
- dataset + subject identity
- dataset + subject identity + classifier

The key comparisons are:

1. How much variance does classifier add after dataset?
2. How much variance does subject quality add after dataset?
3. How much variance does classifier add after dataset + subject quality?
4. How much variance does subject quality add after dataset + classifier?
5. How much variance does subject identity add after dataset?

Subject identity is not a deployable predictor for new subjects, but it quantifies total subject-to-subject variation.

In [ ]:
def build_design_matrix(df: pd.DataFrame, terms: list[str]) -> pd.DataFrame:
    pieces = []

    for term in terms:
        if term == "dataset":
            pieces.append(
                pd.get_dummies(df["dataset_label"], prefix="dataset", drop_first=True, dtype=float)
            )

        elif term == "classifier":
            pieces.append(
                pd.get_dummies(df["model"].astype(str), prefix="model", drop_first=True, dtype=float)
            )

        elif term == "subject_identity":
            pieces.append(
                pd.get_dummies(df["subject_key"].astype(str), prefix="subject", drop_first=True, dtype=float)
            )

        elif term == "subject_quality":
            pieces.append(
                pd.DataFrame({
                    "subject_quality_index": pd.to_numeric(
                        df["subject_quality_index"],
                        errors="coerce",
                    )
                }, index=df.index)
            )

        else:
            raise ValueError(f"Unknown term: {term}")

    if not pieces:
        return pd.DataFrame(index=df.index)

    X = pd.concat(pieces, axis=1)
    X = X.apply(pd.to_numeric, errors="coerce")
    return X


def fit_ols_r2(df: pd.DataFrame, terms: list[str], target_col: str = "auc") -> dict:
    use_cols = [target_col]

    if "subject_quality" in terms:
        use_cols.append("subject_quality_index")

    temp = df.copy()
    temp[target_col] = pd.to_numeric(temp[target_col], errors="coerce")

    for col in use_cols:
        temp = temp[temp[col].notna()].copy()

    y = temp[target_col].values.astype(float)

    X = build_design_matrix(temp, terms)

    if X.shape[1] > 0:
        valid = X.notna().all(axis=1)
        X = X.loc[valid]
        y = y[valid.values]

    n = len(y)

    if n < 5:
        return {
            "terms": "+".join(terms) if terms else "intercept_only",
            "n": n,
            "p": X.shape[1],
            "r2": np.nan,
            "adjusted_r2": np.nan,
            "rmse": np.nan,
        }

    X_np = X.values.astype(float) if X.shape[1] > 0 else np.empty((n, 0))

    # Add intercept
    X_design = np.column_stack([np.ones(n), X_np])

    beta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
    y_hat = X_design @ beta

    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))

    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    p = X.shape[1]
    if n - p - 1 > 0 and np.isfinite(r2):
        adjusted_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1)
    else:
        adjusted_r2 = np.nan

    rmse = float(np.sqrt(np.mean((y - y_hat) ** 2)))

    return {
        "terms": "+".join(terms) if terms else "intercept_only",
        "n": n,
        "p": p,
        "r2": r2,
        "adjusted_r2": adjusted_r2,
        "rmse": rmse,
    }

In [ ]:
model_specs = {
    "intercept_only": [],
    "dataset_only": ["dataset"],
    "classifier_only": ["classifier"],
    "subject_quality_only": ["subject_quality"],
    "dataset_plus_classifier": ["dataset", "classifier"],
    "dataset_plus_subject_quality": ["dataset", "subject_quality"],
    "dataset_plus_classifier_plus_subject_quality": ["dataset", "classifier", "subject_quality"],
    "dataset_plus_subject_identity": ["dataset", "subject_identity"],
    "dataset_plus_subject_identity_plus_classifier": ["dataset", "subject_identity", "classifier"],
}

r2_rows = []

for setting, setting_df in auc_quality_df.groupby("evaluation_setting"):
    for spec_name, terms in model_specs.items():
        result = fit_ols_r2(setting_df, terms=terms, target_col="auc")
        result["evaluation_setting"] = setting
        result["model_spec"] = spec_name
        r2_rows.append(result)

r2_model_comparison_df = pd.DataFrame(r2_rows)

r2_model_comparison_df = r2_model_comparison_df[[
    "evaluation_setting",
    "model_spec",
    "terms",
    "n",
    "p",
    "r2",
    "adjusted_r2",
    "rmse",
]]

save_dataframe(
    r2_model_comparison_df,
    TABLE_DIR / "r2_model_comparison_subject_quality_classifier.csv",
)

display(round_numeric_columns(r2_model_comparison_df, 4))

In [ ]:
increment_rows = []

for setting in r2_model_comparison_df["evaluation_setting"].unique():
    sub = r2_model_comparison_df[
        r2_model_comparison_df["evaluation_setting"] == setting
        ].set_index("model_spec")

    def r2(spec):
        return sub.loc[spec, "r2"] if spec in sub.index else np.nan

    def ar2(spec):
        return sub.loc[spec, "adjusted_r2"] if spec in sub.index else np.nan

    dataset_r2 = r2("dataset_only")
    dataset_classifier_r2 = r2("dataset_plus_classifier")
    dataset_quality_r2 = r2("dataset_plus_subject_quality")
    dataset_classifier_quality_r2 = r2("dataset_plus_classifier_plus_subject_quality")
    dataset_subject_r2 = r2("dataset_plus_subject_identity")
    dataset_subject_classifier_r2 = r2("dataset_plus_subject_identity_plus_classifier")

    dataset_ar2 = ar2("dataset_only")
    dataset_classifier_ar2 = ar2("dataset_plus_classifier")
    dataset_quality_ar2 = ar2("dataset_plus_subject_quality")
    dataset_classifier_quality_ar2 = ar2("dataset_plus_classifier_plus_subject_quality")
    dataset_subject_ar2 = ar2("dataset_plus_subject_identity")
    dataset_subject_classifier_ar2 = ar2("dataset_plus_subject_identity_plus_classifier")

    increment_rows.extend([
        {
            "evaluation_setting": setting,
            "comparison": "classifier_added_after_dataset",
            "r2_increment": dataset_classifier_r2 - dataset_r2,
            "adjusted_r2_increment": dataset_classifier_ar2 - dataset_ar2,
        },
        {
            "evaluation_setting": setting,
            "comparison": "subject_quality_added_after_dataset",
            "r2_increment": dataset_quality_r2 - dataset_r2,
            "adjusted_r2_increment": dataset_quality_ar2 - dataset_ar2,
        },
        {
            "evaluation_setting": setting,
            "comparison": "classifier_added_after_dataset_plus_subject_quality",
            "r2_increment": dataset_classifier_quality_r2 - dataset_quality_r2,
            "adjusted_r2_increment": dataset_classifier_quality_ar2 - dataset_quality_ar2,
        },
        {
            "evaluation_setting": setting,
            "comparison": "subject_quality_added_after_dataset_plus_classifier",
            "r2_increment": dataset_classifier_quality_r2 - dataset_classifier_r2,
            "adjusted_r2_increment": dataset_classifier_quality_ar2 - dataset_classifier_ar2,
        },
        {
            "evaluation_setting": setting,
            "comparison": "subject_identity_added_after_dataset",
            "r2_increment": dataset_subject_r2 - dataset_r2,
            "adjusted_r2_increment": dataset_subject_ar2 - dataset_ar2,
        },
        {
            "evaluation_setting": setting,
            "comparison": "classifier_added_after_dataset_plus_subject_identity",
            "r2_increment": dataset_subject_classifier_r2 - dataset_subject_r2,
            "adjusted_r2_increment": dataset_subject_classifier_ar2 - dataset_subject_ar2,
        },
    ])

incremental_r2_df = pd.DataFrame(increment_rows)

save_dataframe(
    incremental_r2_df,
    TABLE_DIR / "incremental_r2_subject_quality_vs_classifier.csv",
)

display(round_numeric_columns(incremental_r2_df, 4))

## Analysis 5 — Optional Grouped Cross-Validated R²

The previous R² analysis is descriptive/in-sample.

This section adds a simple grouped cross-validation check using subject groups.

It compares only deployable predictors:

- dataset + classifier
- dataset + subject quality
- dataset + classifier + subject quality

Subject identity is not included here because it cannot generalize to unseen subjects.

In [ ]:
def fit_predict_ols_train_test(train_df, test_df, terms, target_col="auc"):
    train = train_df.copy()
    test = test_df.copy()

    train[target_col] = pd.to_numeric(train[target_col], errors="coerce")
    test[target_col] = pd.to_numeric(test[target_col], errors="coerce")

    if "subject_quality" in terms:
        train = train[train["subject_quality_index"].notna()].copy()
        test = test[test["subject_quality_index"].notna()].copy()

    y_train = train[target_col].values.astype(float)
    y_test = test[target_col].values.astype(float)

    X_train = build_design_matrix(train, terms)
    X_test = build_design_matrix(test, terms)

    # Align dummy columns
    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0.0)

    train_valid = X_train.notna().all(axis=1)
    test_valid = X_test.notna().all(axis=1)

    X_train = X_train.loc[train_valid]
    y_train = y_train[train_valid.values]

    X_test = X_test.loc[test_valid]
    y_test = y_test[test_valid.values]

    if len(y_train) < 5 or len(y_test) < 1:
        return np.array([]), np.array([])

    X_train_np = X_train.values.astype(float)
    X_test_np = X_test.values.astype(float)

    X_train_design = np.column_stack([np.ones(len(y_train)), X_train_np])
    X_test_design = np.column_stack([np.ones(len(y_test)), X_test_np])

    beta, *_ = np.linalg.lstsq(X_train_design, y_train, rcond=None)
    y_pred = X_test_design @ beta

    return y_test, y_pred


def grouped_cv_r2(df, terms, n_splits=5):
    temp = df.copy()
    temp = temp.dropna(subset=["auc", "subject_key"]).copy()

    unique_subjects = np.array(sorted(temp["subject_key"].unique()))

    if len(unique_subjects) < n_splits:
        n_splits = max(2, len(unique_subjects))

    rng = np.random.default_rng(42)
    shuffled_subjects = unique_subjects.copy()
    rng.shuffle(shuffled_subjects)

    folds = np.array_split(shuffled_subjects, n_splits)

    all_y = []
    all_pred = []

    for fold_subjects in folds:
        test_mask = temp["subject_key"].isin(fold_subjects)
        train_df = temp[~test_mask].copy()
        test_df = temp[test_mask].copy()

        y_test, y_pred = fit_predict_ols_train_test(
            train_df,
            test_df,
            terms=terms,
            target_col="auc",
        )

        if len(y_test):
            all_y.append(y_test)
            all_pred.append(y_pred)

    if not all_y:
        return {
            "n": 0,
            "cv_r2": np.nan,
            "cv_rmse": np.nan,
        }

    y = np.concatenate(all_y)
    pred = np.concatenate(all_pred)

    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))

    cv_r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    cv_rmse = float(np.sqrt(np.mean((y - pred) ** 2)))

    return {
        "n": len(y),
        "cv_r2": cv_r2,
        "cv_rmse": cv_rmse,
    }

In [ ]:
cv_specs = {
    "dataset_plus_classifier": ["dataset", "classifier"],
    "dataset_plus_subject_quality": ["dataset", "subject_quality"],
    "dataset_plus_classifier_plus_subject_quality": ["dataset", "classifier", "subject_quality"],
}

cv_rows = []

for setting, setting_df in auc_quality_df.groupby("evaluation_setting"):
    for spec_name, terms in cv_specs.items():
        result = grouped_cv_r2(setting_df, terms=terms, n_splits=5)
        result["evaluation_setting"] = setting
        result["model_spec"] = spec_name
        result["terms"] = "+".join(terms)
        cv_rows.append(result)

cv_r2_df = pd.DataFrame(cv_rows)

save_dataframe(
    cv_r2_df,
    TABLE_DIR / "grouped_cv_r2_subject_quality_classifier.csv",
)

display(round_numeric_columns(cv_r2_df, 4))

## Analysis 6 — Simple Figures

These figures are optional, but useful for thesis/results discussion.

They are intentionally simple:

1. subject vs classifier spread
2. incremental R²
3. quality index vs AUC

In [ ]:
plot_df = overall_spread_df.copy()

plot_df = plot_df[[
    "evaluation_setting",
    "subject_auc_z_sd",
    "classifier_auc_z_sd",
]].melt(
    id_vars="evaluation_setting",
    var_name="source",
    value_name="z_auc_sd",
)

plot_df["source"] = plot_df["source"].replace({
    "subject_auc_z_sd": "Subject variability",
    "classifier_auc_z_sd": "Classifier variability",
})

pivot = plot_df.pivot(
    index="evaluation_setting",
    columns="source",
    values="z_auc_sd",
)

ax = pivot.plot(kind="bar", figsize=(8, 5))
ax.set_title("Subject vs Classifier Variability in AUC")
ax.set_xlabel("Evaluation setting")
ax.set_ylabel("SD of dataset-z-scored AUC")
ax.legend(title="")
plt.xticks(rotation=0)
plt.tight_layout()

path = FIGURE_DIR / "subject_vs_classifier_variability_z_auc_sd.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", path)

In [ ]:
plot_df = incremental_r2_df[
    incremental_r2_df["comparison"].isin([
        "classifier_added_after_dataset",
        "subject_quality_added_after_dataset",
        "classifier_added_after_dataset_plus_subject_quality",
        "subject_quality_added_after_dataset_plus_classifier",
    ])
].copy()

plot_df["comparison_label"] = plot_df["comparison"].replace({
    "classifier_added_after_dataset": "Classifier after dataset",
    "subject_quality_added_after_dataset": "Quality after dataset",
    "classifier_added_after_dataset_plus_subject_quality": "Classifier after dataset+quality",
    "subject_quality_added_after_dataset_plus_classifier": "Quality after dataset+classifier",
})

pivot = plot_df.pivot(
    index="comparison_label",
    columns="evaluation_setting",
    values="r2_increment",
)

ax = pivot.plot(kind="bar", figsize=(10, 5))
ax.set_title("Incremental R²: Classifier Choice vs Subject Quality")
ax.set_xlabel("")
ax.set_ylabel("Incremental R²")
ax.legend(title="Evaluation setting")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

path = FIGURE_DIR / "incremental_r2_classifier_vs_subject_quality.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", path)

In [ ]:
plot_df = auc_quality_df.dropna(subset=["subject_quality_index", "auc"]).copy()

plot_df["model_label"] = plot_df["model"].astype(str).map(MODEL_LABELS).fillna(plot_df["model"].astype(str))

# Use dataset-zscored values for the plot
plot_df["quality_z"] = plot_df.groupby("dataset_label")["subject_quality_index"].transform(zscore_within_group)
plot_df["auc_z"] = plot_df.groupby("dataset_label")["auc"].transform(zscore_within_group)

ax = plt.figure(figsize=(7, 5)).gca()

for model_label, model_df in plot_df.groupby("model_label"):
    ax.scatter(
        model_df["quality_z"],
        model_df["auc_z"],
        alpha=0.7,
        label=model_label,
    )

ax.set_title("Subject Quality Index vs AUC")
ax.set_xlabel("Subject quality index, z-scored within dataset")
ax.set_ylabel("AUC, z-scored within dataset")
ax.legend(title="Model")
plt.tight_layout()

path = FIGURE_DIR / "subject_quality_index_vs_auc_zscored.png"
plt.savefig(path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", path)

## Final Compact Summary Tables

These are the tables most useful for reporting.

In [ ]:
# Pull the most important values into one compact table.

summary_rows = []

for setting in ["within_subject", "loso"]:
    spread_row = overall_spread_df[
        overall_spread_df["evaluation_setting"] == setting
        ].iloc[0]

    corr_row = cross_classifier_corr_summary_df[
        cross_classifier_corr_summary_df["evaluation_setting"] == setting
        ].iloc[0]

    inc_sub = incremental_r2_df[
        incremental_r2_df["evaluation_setting"] == setting
        ].set_index("comparison")

    def get_inc(name):
        if name in inc_sub.index:
            return inc_sub.loc[name, "r2_increment"]
        return np.nan

    summary_rows.append({
        "evaluation_setting": setting,
        "n_subjects": int(spread_row["n_subjects"]),
        "subject_auc_z_sd": spread_row["subject_auc_z_sd"],
        "classifier_auc_z_sd": spread_row["classifier_auc_z_sd"],
        "sd_ratio_subject_over_classifier": spread_row["sd_ratio_subject_over_classifier"],
        "subject_auc_z_range": spread_row["subject_auc_z_range"],
        "classifier_auc_z_range": spread_row["classifier_auc_z_range"],
        "range_ratio_subject_over_classifier": spread_row["range_ratio_subject_over_classifier"],
        "mean_cross_classifier_subject_auc_rho": corr_row["mean_spearman_rho"],
        "median_cross_classifier_subject_auc_rho": corr_row["median_spearman_rho"],
        "r2_classifier_added_after_dataset": get_inc("classifier_added_after_dataset"),
        "r2_subject_quality_added_after_dataset": get_inc("subject_quality_added_after_dataset"),
        "r2_classifier_added_after_dataset_plus_quality": get_inc("classifier_added_after_dataset_plus_subject_quality"),
        "r2_quality_added_after_dataset_plus_classifier": get_inc("subject_quality_added_after_dataset_plus_classifier"),
        "r2_subject_identity_added_after_dataset": get_inc("subject_identity_added_after_dataset"),
        "r2_classifier_added_after_dataset_plus_subject_identity": get_inc("classifier_added_after_dataset_plus_subject_identity"),
    })

relative_importance_final_summary_df = pd.DataFrame(summary_rows)

save_dataframe(
    relative_importance_final_summary_df,
    TABLE_DIR / "relative_importance_final_summary.csv",
)

display(round_numeric_columns(relative_importance_final_summary_df, 4))

In [ ]:
for _, row in relative_importance_final_summary_df.iterrows():
    setting = row["evaluation_setting"]

    print("=" * 100)
    print(setting)
    print("=" * 100)

    print(
        f"For {setting}, the SD of subject-level mean AUCs after dataset z-scoring was "
        f"{row['subject_auc_z_sd']:.3f}, while the SD of classifier-level mean AUCs was "
        f"{row['classifier_auc_z_sd']:.3f}. "
        f"This gives a subject/classifier variability ratio of "
        f"{row['sd_ratio_subject_over_classifier']:.2f}."
    )

    print(
        f"The mean cross-classifier Spearman correlation of subject AUCs was "
        f"{row['mean_cross_classifier_subject_auc_rho']:.3f} "
        f"(median = {row['median_cross_classifier_subject_auc_rho']:.3f}), "
        f"which indicates how consistently easy/difficult subjects are across classifiers."
    )

    print(
        f"In incremental R² terms, classifier identity added "
        f"{row['r2_classifier_added_after_dataset']:.3f} after dataset, while the "
        f"subject-quality index added {row['r2_subject_quality_added_after_dataset']:.3f} "
        f"after dataset."
    )

    print(
        f"After accounting for both dataset and subject quality, classifier identity added "
        f"{row['r2_classifier_added_after_dataset_plus_quality']:.3f}. "
        f"After accounting for dataset and classifier, subject quality added "
        f"{row['r2_quality_added_after_dataset_plus_classifier']:.3f}."
    )

    print(
        f"Subject identity, which captures all subject-specific factors, added "
        f"{row['r2_subject_identity_added_after_dataset']:.3f} after dataset. "
        f"After subject identity was included, classifier identity added only "
        f"{row['r2_classifier_added_after_dataset_plus_subject_identity']:.3f}."
    )

    print()

## Interpretation Guidance

The strongest possible conclusion should be phrased carefully.

Good wording:

> In these datasets, subject-to-subject variability and subject/signal quality appear to explain more variation in decoding performance than classifier identity alone. This is supported by larger subject-level AUC variability than classifier-level variability, cross-classifier consistency in which subjects are easy or difficult, and incremental R² analyses showing that subject quality/identity accounts for substantial AUC variation.

Avoid overclaiming:

> Subject quality is always more important than classifier type in EEG decoding.

A safer thesis statement:

> In the present oddball/ERP datasets, classifier choice matters, especially when comparing within-subject and LOSO settings, but subject/signal quality appears to be a larger determinant of decoding success than the specific classifier among the three tested models.

In [ ]:
output_files = sorted(TABLE_DIR.glob("*.csv"))

inventory_df = pd.DataFrame({
    "file": [p.name for p in output_files],
    "path": [str(p) for p in output_files],
})

save_dataframe(
    inventory_df,
    TABLE_DIR / "relative_importance_output_inventory.csv",
)

display(inventory_df)

In [ ]:
print("Notebook 24 completed.")
print("Main outputs saved in:", TABLE_DIR)
print()
print("Most important tables:")
print("-", TABLE_DIR / "relative_importance_final_summary.csv")
print("-", TABLE_DIR / "subject_vs_classifier_spread_summary.csv")
print("-", TABLE_DIR / "overall_subject_vs_classifier_spread_zscored_summary.csv")
print("-", TABLE_DIR / "cross_classifier_subject_auc_correlations.csv")
print("-", TABLE_DIR / "subject_quality_index_auc_correlations.csv")
print("-", TABLE_DIR / "r2_model_comparison_subject_quality_classifier.csv")
print("-", TABLE_DIR / "incremental_r2_subject_quality_vs_classifier.csv")
print("-", TABLE_DIR / "grouped_cv_r2_subject_quality_classifier.csv")
print()
print("Most important figures:")
print("-", FIGURE_DIR / "subject_vs_classifier_variability_z_auc_sd.png")
print("-", FIGURE_DIR / "incremental_r2_classifier_vs_subject_quality.png")
print("-", FIGURE_DIR / "subject_quality_index_vs_auc_zscored.png")